## 各モデルでMACsのNLOpsがどれくらいの量なのかを試算
Liftタスク input_dim = 19, output_dim = 7
### MLP 1024×1024, Layer2
#### MACs
(input_dim × hidden_dim)+(hidden_dim × hidden_dim) + (hidden_dim × output_dim) = 1075200 ≒ 1.07M
#### NLOps
hidden_dim × 2 + output_dim = 2055 ≒ 2K

### RNN 400×400, Layer2, LSTM
#### MACs
```
    macs_input = 4 * (input_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)  # +bias
    macs_deeper = 4 * (hidden_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)
    macs_output = hidden_dim * output_dim + output_dim 

    macs_per_step = macs_input + macs_ deeper * (n_layer - 1) + macs_output
```
= 1962807 ≒ 1.96M
#### NLOps
```
    nlops_per_layer = (3 + 2) * hidden_dim  
    nlops_per_step = num_layers * nlops_per_layer + output_dim
```
= 4007 ≒ 4K

### NCP 128units, ode_unfolds:3
#### MACs
```
    macs_constant = 4
    macs_MVM = units * units 
    macs_per_step = (macs_constant + macs_MVM * 2) * units * ode_unfolds
```
= 12584448 ≒ 12.6M  
#### NLOps
```
    nlops_per_step = ode_unfolds * (units * units + units) + input_dim
```
= 49559 ≒ 49K
### NCP 256units, ode_unfolds:3
#### MACs
= 100666368 ≒ 100M
#### NLOps
= 197395 ≒ 197K


In [2]:
input_dim = 19
output_dim = 7
hidden_dim = 1024
macs=(input_dim * hidden_dim)+(hidden_dim * hidden_dim) + (hidden_dim * output_dim)
nlops = hidden_dim * 2 + output_dim
print("MACs:", macs, "NLOps:", nlops)

MACs: 1075200 NLOps: 2055


In [3]:
def lstm_macs_nlops(input_dim: int, hidden_dim: int, num_layers: int, output_dim: int = 0):
    """
    戻り値:
      macs_per_step: 1ステップの乗算加算回数（近似）
      nlops_per_step: 1ステップの非線形演算回数（近似）
    前提:
      - LSTMは4ゲート (i, f, g, o)
      - 各ゲート: x→W_x、h→W_h の線形 + バイアス
      - 非線形は: sigmoid×3 (i,f,o) + tanh×1 (g) + tanh×1（セル出力）
    """
    # 1層目の入力は input_dim、以降の層の入力は hidden_dim
    macs_input_layer = 4 * (input_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)  # +bias
    macs_deeper_layer = 4 * (hidden_dim * hidden_dim + hidden_dim * hidden_dim + hidden_dim)
    macs_lstm_all_layers = macs_input_layer + (num_layers - 1) * macs_deeper_layer

    # 出力線形（必要なら）
    macs_output = 0
    if output_dim and output_dim > 0:
        macs_output = hidden_dim * output_dim + output_dim  # +bias

    macs_per_step = macs_lstm_all_layers + macs_output

    # 非線形演算数（1層あたり）
    # 各ゲート: sigmoid×3・tanh×1、さらにセル出力のtanh×1 → 合計 tanh×2 + sigmoid×3
    # hidden_dim 要素に対して適用されるとみなす
    nlops_per_layer = (3 + 2) * hidden_dim  # sigmoid3 + tanh2
    nlops_per_step = num_layers * nlops_per_layer
    # 出力でtanhを使う設定なら +output_dim などを加算（ここでは通常は線形出力と仮定して0）

    return macs_per_step, nlops_per_step

# 例: input_dim=23, hidden_dim=400, num_layers=2, output_dim=7
macs, nlops = lstm_macs_nlops(input_dim=23, hidden_dim=400, num_layers=2, output_dim=7)
print("Per-step MACs:", macs, "Per-step NLOps:", nlops)

Per-step MACs: 1962807 Per-step NLOps: 4000


In [8]:
def ncp_macs_nlops(input_dim: int, units: int, output_dim: int, ode_unfolds: int,
                   k_nl: int = 1):
    """
    NCP(LTC/CfC)の1ステップあたりのMACs/NLOpsを上限寄りで近似。
    - k_nl: 1ユニットに対する非線形の個数係数（tanh, sigmoid, softplus等の合計）
            LTCなら3〜5程度、CfCでも同程度で近似することが多い。
    """
    macs_core_per_unfold = (units * units) * 2 + 4
    nlops_core_per_unfold = units * units

    macs_per_step = ode_unfolds * macs_core_per_unfold * units
    nlops_per_step = ode_unfolds * (nlops_core_per_unfold + units) + input_dim

    return macs_per_step, nlops_per_step

# 例: input_dim=23, units=128, output_dim=7, ode_unfolds=3
macs, nlops = ncp_macs_nlops(input_dim=19, units=256, output_dim=7, ode_unfolds=3, k_nl=1)
print("Per-step MACs (approx):", macs, "Per-step NLOps (approx):", nlops)

Per-step MACs (approx): 100666368 Per-step NLOps (approx): 197395
